# 04 — Cell Type Annotation (Integrated Space)

Reannotates cell types in the Harmony-integrated object. The pre-integration annotations from colab_02 were assigned per-dataset; here we annotate the 19 integrated Leiden clusters using marker genes detected in the shared embedding.

**Tasks:**
1. Re-add `gestational_week` (Zhong) and `sample` (Bhaduri) — dropped during `ad.concat` in colab_03
2. Detect marker genes per integrated Leiden cluster (Wilcoxon rank-sum)
3. Score clusters against known brain cell type marker panels
4. Assign cell type labels to all 19 clusters
5. Compare organoid vs fetal composition per cell type

| Item | Value |
|------|-------|
| Input | `integrated_harmony.h5ad` (226,659 cells × 2,000 HVGs; raw: 14,498 genes) |
| Leiden clusters | 19 (resolution=0.5) |
| Datasets | Bhaduri 2020 (organoids) + Zhong 2018 (fetal PFC) |

**Prerequisites:** `colab_03_integration.ipynb` must have been run. `integrated_harmony.h5ad` must be on Drive.

**Runtime:** Standard or High-RAM — this notebook is lightweight (no recomputation of PCA/UMAP/Harmony).

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install scanpy and leidenalg
!pip install -q scanpy leidenalg

In [ ]:
# Import libraries and define all input/output paths
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

PATHS = {
    'integrated':        os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_harmony.h5ad'),
    'bhaduri_clustered': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad'),
    'zhong_clustered':   os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_clustered.h5ad'),
    'annotated':         os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_annotated.h5ad'),
}

print('Paths configured.')
for k, v in PATHS.items():
    print(f'  {k}: {v}')

## 2. Load Integrated Object

In [ ]:
# Load the Harmony-integrated object from colab_03
adata = sc.read_h5ad(PATHS['integrated'])

print(f'Integrated object: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes (HVGs)')
print(f'Raw layer:         {adata.raw.shape[0]:,} cells x {adata.raw.shape[1]:,} genes (all shared)')
print()
print('obs columns:', list(adata.obs.columns))
print('obsm keys:  ', list(adata.obsm.keys()))
print()
print('Leiden clusters:', adata.obs['leiden'].nunique())
print('Datasets:', adata.obs['dataset'].value_counts().to_dict())

## 3. Re-add Dataset-Specific Metadata

`ad.concat` in colab_03 dropped columns that only existed in one dataset:
- `gestational_week` — Zhong only (extracted from barcodes in colab_02)
- `sample` — Bhaduri only (extracted from barcodes in colab_02)

We reload these from the individual clustered h5ads and map them back using the barcode prefixes (`bhaduri_` / `zhong_`) that were added before concatenation.

In [ ]:
# Load only the obs dataframes from the individual clustered h5ads (not the full objects)
import anndata as ad

# Read just the obs from each file — backed mode avoids loading X into RAM
adata_bhaduri_backed = sc.read_h5ad(PATHS['bhaduri_clustered'], backed='r')
adata_zhong_backed   = sc.read_h5ad(PATHS['zhong_clustered'], backed='r')

obs_bhaduri = adata_bhaduri_backed.obs.copy()
obs_zhong   = adata_zhong_backed.obs.copy()

# Close backed objects to free file handles
del adata_bhaduri_backed, adata_zhong_backed

print('Bhaduri obs columns:', list(obs_bhaduri.columns))
print('Zhong obs columns:  ', list(obs_zhong.columns))
print()
print(f'Bhaduri obs: {len(obs_bhaduri):,} cells')
print(f'Zhong obs:   {len(obs_zhong):,} cells')

In [ ]:
# Re-add 'sample' column for Bhaduri cells
# Integrated barcodes are prefixed: 'bhaduri_ORIGINALBARCODE'
# Map: strip prefix → look up in obs_bhaduri → assign

if 'sample' in obs_bhaduri.columns:
    sample_map = obs_bhaduri['sample'].to_dict()  # original_barcode → sample
    
    bhaduri_mask = adata.obs['dataset'] == 'bhaduri'
    original_barcodes = adata.obs_names[bhaduri_mask].str.removeprefix('bhaduri_')
    
    adata.obs['sample'] = pd.Series(index=adata.obs_names, dtype='object')
    adata.obs.loc[bhaduri_mask, 'sample'] = [
        sample_map.get(bc, np.nan) for bc in original_barcodes
    ]
    
    n_mapped = adata.obs.loc[bhaduri_mask, 'sample'].notna().sum()
    print(f'sample: mapped {n_mapped:,} / {bhaduri_mask.sum():,} Bhaduri cells')
    print(f'Unique samples: {adata.obs["sample"].dropna().nunique()}')
    print(adata.obs['sample'].value_counts().head(10))
else:
    print('WARNING: sample column not found in Bhaduri clustered h5ad')

In [ ]:
# Re-add 'gestational_week' column for Zhong cells

if 'gestational_week' in obs_zhong.columns:
    gw_map = obs_zhong['gestational_week'].to_dict()  # original_barcode → GW
    
    zhong_mask = adata.obs['dataset'] == 'zhong'
    original_barcodes = adata.obs_names[zhong_mask].str.removeprefix('zhong_')
    
    adata.obs['gestational_week'] = pd.Series(index=adata.obs_names, dtype='object')
    adata.obs.loc[zhong_mask, 'gestational_week'] = [
        gw_map.get(bc, np.nan) for bc in original_barcodes
    ]
    
    n_mapped = adata.obs.loc[zhong_mask, 'gestational_week'].notna().sum()
    print(f'gestational_week: mapped {n_mapped:,} / {zhong_mask.sum():,} Zhong cells')
    print(adata.obs['gestational_week'].value_counts().sort_index())
else:
    print('WARNING: gestational_week column not found in Zhong clustered h5ad')

In [ ]:
# Clean up — free the obs dataframes
del obs_bhaduri, obs_zhong, sample_map, gw_map

import gc
gc.collect()

print('Updated obs columns:', list(adata.obs.columns))

## 4. Quick Orientation — Integrated UMAP

Before diving into markers, refresh our view of the integrated embedding.

In [ ]:
# Integrated UMAP colored by Leiden cluster and dataset
sc.pl.umap(adata, color='leiden', legend_loc='on data',
           title='Integrated Leiden clusters (res=0.5)')

sc.pl.umap(adata, color='dataset', title='Integrated — by dataset')

In [ ]:
# Cluster sizes — helps calibrate expectations for marker gene detection
print('Cluster sizes:')
print(adata.obs['leiden'].value_counts().sort_index().to_string())
print()
print(f'Total cells: {adata.shape[0]:,}')

## 5. Marker Gene Detection

Wilcoxon rank-sum test: for each cluster, finds genes significantly upregulated compared to all other cells. Uses `use_raw=True` to pull from the log-normalized full gene set (14,498 genes), not just the 2,000 HVGs.

This is the primary evidence for assigning cell type identities.

In [ ]:
# Detect differentially expressed genes per Leiden cluster
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', use_raw=True)
print('Marker gene detection complete.')

In [ ]:
# Plot top 5 marker genes per cluster — visual overview
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# Print top 15 marker genes per cluster as text — for detailed inspection
result = adata.uns['rank_genes_groups']
n_top = 15

for cluster in adata.obs['leiden'].cat.categories:
    idx = int(cluster)
    genes  = [result['names'][i][idx] for i in range(n_top)]
    scores = [result['scores'][i][idx] for i in range(n_top)]
    pvals  = [result['pvals_adj'][i][idx] for i in range(n_top)]
    
    print(f'\n--- Cluster {cluster} ({adata.obs["leiden"].value_counts()[cluster]:,} cells) ---')
    for g, s, p in zip(genes, scores, pvals):
        print(f'  {g:15s}  score={s:7.1f}  padj={p:.2e}')

## 6. Known Marker Panel — Dotplot

Systematic check of canonical brain cell type markers across all integrated clusters. Dot size = fraction of cells expressing the gene; color = mean expression level.

**Marker categories:**
- **Radial glia / vRG:** SOX2, PAX6, HES1, NES, VIM
- **Outer radial glia (oRG):** HOPX, FAM107A, TNC
- **Intermediate progenitors (IP):** EOMES, PPP1R17
- **Cycling:** MKI67, TOP2A
- **Excitatory neurons (general):** NEUROD2, NEUROD6, SLC17A7
- **Deep layer excitatory:** TBR1, BCL11B (CTIP2)
- **Upper layer excitatory:** SATB2, CUX1, CUX2
- **Immature / migrating neurons:** DCX, STMN2
- **GABAergic (general):** GAD1, GAD2, DLX1, DLX2, DLX5
- **MGE interneurons:** SST, LHX6
- **CGE interneurons:** CALB2, VIP, SP8
- **Astrocytes:** GFAP, AQP4, S100B
- **OPCs:** OLIG1, OLIG2, PDGFRA
- **Microglia:** AIF1, CSF1R, CX3CR1
- **Stress / immediate early genes:** FOS, JUN, HSPA1A, HSPA1B

In [ ]:
# Define the full marker panel organized by cell type category
MARKER_PANEL = {
    'Radial glia':       ['SOX2', 'PAX6', 'HES1', 'NES', 'VIM'],
    'oRG':               ['HOPX', 'FAM107A', 'TNC'],
    'IP':                ['EOMES', 'PPP1R17'],
    'Cycling':           ['MKI67', 'TOP2A'],
    'Excitatory (gen.)': ['NEUROD2', 'NEUROD6', 'SLC17A7'],
    'Deep layer exc.':   ['TBR1', 'BCL11B'],
    'Upper layer exc.':  ['SATB2', 'CUX1', 'CUX2'],
    'Immature neurons':  ['DCX', 'STMN2'],
    'GABAergic':         ['GAD1', 'GAD2', 'DLX1', 'DLX2', 'DLX5'],
    'MGE interneurons':  ['SST', 'LHX6'],
    'CGE interneurons':  ['CALB2', 'VIP', 'SP8'],
    'Astrocytes':        ['GFAP', 'AQP4', 'S100B'],
    'OPCs':              ['OLIG1', 'OLIG2', 'PDGFRA'],
    'Microglia':         ['AIF1', 'CSF1R', 'CX3CR1'],
    'Stress':            ['FOS', 'JUN', 'HSPA1A', 'HSPA1B'],
}

# Check which markers are available in adata.raw
all_markers = [g for genes in MARKER_PANEL.values() for g in genes]
available = [g for g in all_markers if g in adata.raw.var_names]
missing   = [g for g in all_markers if g not in adata.raw.var_names]

print(f'Available: {len(available)} / {len(all_markers)} markers')
if missing:
    print(f'Missing from shared gene set: {missing}')

# Build the filtered panel (only available genes) for the dotplot
marker_panel_available = {}
for cat, genes in MARKER_PANEL.items():
    present = [g for g in genes if g in adata.raw.var_names]
    if present:
        marker_panel_available[cat] = present

print(f'\nCategories with markers: {len(marker_panel_available)}')

In [ ]:
# Dotplot — all clusters vs all marker categories
# This is the single most useful plot for annotation: shows which cluster
# expresses which marker category, making cell type assignment systematic
sc.pl.dotplot(
    adata,
    var_names=marker_panel_available,
    groupby='leiden',
    use_raw=True,
    standard_scale='var',   # scale each gene 0–1 so markers with different expression ranges are comparable
    title='Integrated clusters — known brain markers',
    figsize=(18, 8),
)

## 7. Feature Plots — Key Markers on UMAP

Complements the dotplot: shows where each marker is expressed spatially on the UMAP. Useful for confirming that marker expression aligns with cluster boundaries.

In [ ]:
# Progenitor markers on UMAP
progenitor_markers = [g for g in ['SOX2', 'PAX6', 'HOPX', 'EOMES', 'MKI67', 'TOP2A']
                      if g in adata.raw.var_names]
sc.pl.umap(adata, color=progenitor_markers, use_raw=True, ncols=3,
           title=[f'{g}' for g in progenitor_markers])

In [ ]:
# Neuronal markers on UMAP
neuronal_markers = [g for g in ['NEUROD2', 'TBR1', 'BCL11B', 'SATB2', 'DCX', 'STMN2']
                    if g in adata.raw.var_names]
sc.pl.umap(adata, color=neuronal_markers, use_raw=True, ncols=3,
           title=[f'{g}' for g in neuronal_markers])

In [ ]:
# Interneuron markers on UMAP
interneuron_markers = [g for g in ['GAD1', 'GAD2', 'DLX2', 'SST', 'LHX6', 'CALB2', 'VIP']
                       if g in adata.raw.var_names]
sc.pl.umap(adata, color=interneuron_markers, use_raw=True, ncols=3,
           title=[f'{g}' for g in interneuron_markers])

In [ ]:
# Glial + stress markers on UMAP
other_markers = [g for g in ['GFAP', 'AQP4', 'OLIG2', 'PDGFRA', 'AIF1', 'CSF1R', 'FOS', 'HSPA1A']
                 if g in adata.raw.var_names]
sc.pl.umap(adata, color=other_markers, use_raw=True, ncols=3,
           title=[f'{g}' for g in other_markers])

## 8. Dataset Composition per Cluster

Before annotating, check which clusters are dataset-specific vs shared. Clusters that are 100% one dataset may represent biology unique to that condition (e.g., stress in organoids, microglia in fetal).

In [ ]:
# Dataset composition per Leiden cluster
cluster_comp = (
    adata.obs.groupby(['leiden', 'dataset'])
    .size()
    .unstack(fill_value=0)
)
cluster_comp_pct = cluster_comp.div(cluster_comp.sum(axis=1), axis=0) * 100

# Add total cell count
cluster_comp_pct['total_cells'] = cluster_comp.sum(axis=1)

print('Dataset composition per cluster (%):')
print(cluster_comp_pct.round(1).to_string())
print()

# Flag dataset-specific clusters
for cl in cluster_comp_pct.index:
    if 'zhong' not in cluster_comp_pct.columns:
        continue
    zhong_pct = cluster_comp_pct.loc[cl, 'zhong'] if 'zhong' in cluster_comp_pct.columns else 0
    bhaduri_pct = cluster_comp_pct.loc[cl, 'bhaduri'] if 'bhaduri' in cluster_comp_pct.columns else 0
    if bhaduri_pct == 100:
        print(f'  Cluster {cl}: 100% Bhaduri (organoid-specific)')
    elif zhong_pct > 5:
        print(f'  Cluster {cl}: {zhong_pct:.1f}% Zhong (enriched fetal contribution)')

## 9. Pre-Integration Cell Type Mapping

The `cell_type` column from colab_03 contains per-dataset annotations assigned before integration. Cross-tabulating these with the new integrated Leiden clusters shows how pre-integration cell types map onto the integrated clustering.

This helps annotation: if integrated cluster 3 is mostly cells that were labeled "oRG" before integration, it's likely still oRG.

In [ ]:
# Cross-tabulation: pre-integration cell_type vs integrated leiden
if 'cell_type' in adata.obs.columns:
    ct_vs_leiden = pd.crosstab(
        adata.obs['leiden'],
        adata.obs['cell_type'],
    )
    
    # Show as percentages (row-normalized: for each integrated cluster, what % came from each pre-integration type)
    ct_vs_leiden_pct = ct_vs_leiden.div(ct_vs_leiden.sum(axis=1), axis=0) * 100
    
    # For each cluster, show the dominant pre-integration cell type(s)
    print('Dominant pre-integration cell types per integrated cluster:')
    print('=' * 80)
    for cl in ct_vs_leiden_pct.index:
        row = ct_vs_leiden_pct.loc[cl].sort_values(ascending=False)
        top = row[row > 5]  # only show types contributing >5%
        top_str = ', '.join([f'{ct} ({pct:.0f}%)' for ct, pct in top.items()])
        n_cells = ct_vs_leiden.loc[cl].sum()
        print(f'  Cluster {cl:>2s} ({n_cells:>6,} cells): {top_str}')
else:
    print('cell_type column not found — skipping cross-tabulation')

## 10. Annotate Integrated Clusters

Based on three lines of evidence:
1. **Top marker genes** (section 5) — what genes define each cluster
2. **Known marker dotplot** (section 6) — which canonical markers are expressed
3. **Pre-integration mapping** (section 9) — what these cells were labeled before integration

**Instructions:** After running the cells above, inspect all the evidence and fill in the `CLUSTER_ANNOTATIONS` dictionary below. Each entry maps a Leiden cluster ID (string) to a cell type label.

Run the annotation cell to assign labels, then proceed to visualization.

In [ ]:
# =====================================================================
# ANNOTATE HERE — fill in after inspecting markers, dotplot, and
# pre-integration mapping above.
#
# Format: 'leiden_cluster_id': 'Cell Type Label'
#
# Guidelines:
#   - Use consistent naming across clusters (e.g., always 'Excitatory neurons',
#     not sometimes 'Exc. neurons' and sometimes 'Excitatory')
#   - Append qualifiers in parentheses for subtypes:
#     'Excitatory neurons (deep layer)', 'Excitatory neurons (upper layer)'
#   - For dataset-specific clusters, note the origin:
#     'Stressed RG (organoid)', 'Microglia (fetal)'
# =====================================================================

CLUSTER_ANNOTATIONS = {
    '0':  '',   # fill in
    '1':  '',   # fill in
    '2':  '',   # fill in
    '3':  '',   # fill in
    '4':  '',   # fill in
    '5':  '',   # fill in
    '6':  '',   # fill in
    '7':  '',   # fill in
    '8':  '',   # fill in
    '9':  '',   # fill in
    '10': '',   # fill in
    '11': '',   # fill in
    '12': '',   # fill in
    '13': '',   # fill in
    '14': '',   # fill in
    '15': '',   # fill in
    '16': '',   # fill in
    '17': '',   # fill in
    '18': '',   # fill in
}

# Validate: all clusters accounted for, no empty labels
leiden_cats = adata.obs['leiden'].cat.categories.tolist()
missing_clusters = [c for c in leiden_cats if c not in CLUSTER_ANNOTATIONS]
empty_labels     = [c for c, v in CLUSTER_ANNOTATIONS.items() if v == '']

if missing_clusters:
    print(f'WARNING: clusters not in annotation dict: {missing_clusters}')
if empty_labels:
    print(f'INCOMPLETE: clusters with empty labels: {empty_labels}')
    print('Fill in all labels above before proceeding.')
else:
    # Apply annotations
    adata.obs['cell_type_integrated'] = adata.obs['leiden'].map(CLUSTER_ANNOTATIONS).astype('category')
    print('Annotations applied to adata.obs["cell_type_integrated"]')
    print()
    print(adata.obs['cell_type_integrated'].value_counts())

## 11. Visualize Annotated UMAP

In [ ]:
# Annotated UMAP — numbered labels with side legend (same style as colab_03)
fig, ax = plt.subplots(figsize=(10, 8))

cell_types = adata.obs['cell_type_integrated'].cat.categories.tolist()
umap_coords = adata.obsm['X_umap']
palette = sc.pl.palettes.default_20 + sc.pl.palettes.default_20  # extend if >20 types

for i, ct in enumerate(cell_types):
    mask = (adata.obs['cell_type_integrated'] == ct).values
    ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
               s=0.3, alpha=0.5, color=palette[i % len(palette)], rasterized=True)

# Number at centroid
for i, ct in enumerate(cell_types):
    mask = (adata.obs['cell_type_integrated'] == ct).values
    cx = umap_coords[mask, 0].mean()
    cy = umap_coords[mask, 1].mean()
    ax.text(cx, cy, str(i + 1), fontsize=8, fontweight='bold', ha='center', va='center',
            color='black', bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

# Side legend
legend_text = '\n'.join([f'{i+1}. {ct}' for i, ct in enumerate(cell_types)])
ax.text(1.02, 0.98, legend_text, transform=ax.transAxes,
        fontsize=7, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('Integrated UMAP — cell type annotation')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: Leiden clusters vs cell type annotations
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc.pl.umap(adata, color='leiden', legend_loc='on data', ax=axes[0], show=False,
           title='Leiden clusters')
sc.pl.umap(adata, color='cell_type_integrated', ax=axes[1], show=False,
           title='Cell type annotations')

plt.tight_layout()
plt.show()

## 12. Validation — Marker Dotplot by Cell Type

Repeat the dotplot from section 6, but grouped by `cell_type_integrated` instead of Leiden cluster. This confirms that the annotations are consistent with marker expression — e.g., clusters labeled "oRG" should express HOPX/FAM107A, not GAD1/GAD2.

In [ ]:
# Dotplot grouped by cell type annotation — validation
sc.pl.dotplot(
    adata,
    var_names=marker_panel_available,
    groupby='cell_type_integrated',
    use_raw=True,
    standard_scale='var',
    title='Cell type annotations — marker validation',
    figsize=(18, 8),
)

## 13. Organoid vs Fetal — Cell Type Composition

The central question: which cell types are shared between organoids and fetal brain, and which are unique to one condition?

**Note on interpretation:** Bhaduri has ~224k cells vs Zhong's ~2.3k. Raw cell counts are not directly comparable — look at the *within-dataset proportions* (what fraction of each dataset's cells belong to each type).

In [ ]:
# Cell type composition per dataset — within-dataset proportions
comp = pd.crosstab(
    adata.obs['cell_type_integrated'],
    adata.obs['dataset'],
)

# Within-dataset percentages (column-normalized)
comp_pct = comp.div(comp.sum(axis=0), axis=1) * 100

# Add raw counts
comp_display = comp_pct.round(1).copy()
comp_display.columns = [f'{c} (%)' for c in comp_display.columns]
for c in comp.columns:
    comp_display[f'{c} (n)'] = comp[c]

# Sort by total cell count
comp_display['total'] = comp.sum(axis=1)
comp_display = comp_display.sort_values('total', ascending=False)

print('Cell type composition — within-dataset proportions:')
print('=' * 90)
print(comp_display.to_string())

In [ ]:
# Stacked bar chart — within-dataset proportions side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, dataset in zip(axes, ['bhaduri', 'zhong']):
    if dataset not in comp_pct.columns:
        continue
    data = comp_pct[dataset].sort_values(ascending=True)
    colors = [palette[i % len(palette)] for i in range(len(data))]
    data.plot.barh(ax=ax, color=colors)
    ax.set_xlabel('% of dataset')
    ax.set_title(f'{dataset.capitalize()} — cell type proportions')
    ax.set_xlim(0, max(comp_pct.max()) * 1.1)
    # Add percentage labels
    for j, (val, name) in enumerate(zip(data.values, data.index)):
        if val > 0.5:  # only label if visible
            ax.text(val + 0.3, j, f'{val:.1f}%', va='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Split UMAP — organoid cells vs fetal cells, colored by cell type
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, dataset in zip(axes, ['bhaduri', 'zhong']):
    mask = (adata.obs['dataset'] == dataset).values
    # Plot background (other dataset) in light grey
    ax.scatter(umap_coords[~mask, 0], umap_coords[~mask, 1],
               s=0.1, alpha=0.05, color='lightgrey', rasterized=True)
    # Plot this dataset colored by cell type
    for i, ct in enumerate(cell_types):
        ct_mask = mask & (adata.obs['cell_type_integrated'] == ct).values
        if ct_mask.sum() > 0:
            ax.scatter(umap_coords[ct_mask, 0], umap_coords[ct_mask, 1],
                       s=0.5, alpha=0.5, color=palette[i % len(palette)],
                       label=ct, rasterized=True)
    ax.set_title(f'{dataset.capitalize()} cells')
    ax.axis('off')

# Shared legend
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', fontsize=7, markerscale=5,
           bbox_to_anchor=(1.0, 0.5))

plt.suptitle('Integrated UMAP — split by dataset', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.88, 0.96])
plt.show()

## 14. Temporal Analysis — Gestational Week

For the Zhong (fetal) cells, check whether cell type composition shifts across gestational weeks (GW8–GW26). Early timepoints should be progenitor-enriched; late timepoints should be neuron-enriched.

In [ ]:
# Gestational week on the integrated UMAP (NaN for Bhaduri cells → grey)
if 'gestational_week' in adata.obs.columns:
    sc.pl.umap(adata, color='gestational_week',
               title='Integrated UMAP — gestational week (Zhong only)')
else:
    print('gestational_week not available')

In [ ]:
# Cell type composition per gestational week (Zhong cells only)
if 'gestational_week' in adata.obs.columns:
    zhong_obs = adata.obs[adata.obs['dataset'] == 'zhong'].copy()
    
    if zhong_obs['gestational_week'].notna().sum() > 0:
        gw_comp = pd.crosstab(
            zhong_obs['gestational_week'],
            zhong_obs['cell_type_integrated'],
        )
        gw_comp_pct = gw_comp.div(gw_comp.sum(axis=1), axis=0) * 100
        
        print('Zhong — cell type composition per gestational week (%):')
        print(gw_comp_pct.round(1).to_string())
        print()
        print('Cell counts per gestational week:')
        print(gw_comp.sum(axis=1))
        
        # Stacked bar chart
        fig, ax = plt.subplots(figsize=(12, 6))
        gw_comp_pct.plot.bar(stacked=True, ax=ax, colormap='tab20')
        ax.set_ylabel('% of cells')
        ax.set_title('Zhong fetal PFC — cell type proportions by gestational week')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print('No gestational_week values mapped for Zhong cells')
else:
    print('gestational_week not available')

## 15. Sample Analysis — Bhaduri Organoid Protocols

Check whether cell type composition varies across Bhaduri's organoid samples/protocols. Protocol-dependent biases would show as certain cell types being over- or under-represented in specific samples.

In [ ]:
# Cell type composition per Bhaduri sample
if 'sample' in adata.obs.columns:
    bhaduri_obs = adata.obs[adata.obs['dataset'] == 'bhaduri'].copy()
    
    if bhaduri_obs['sample'].notna().sum() > 0:
        sample_comp = pd.crosstab(
            bhaduri_obs['sample'],
            bhaduri_obs['cell_type_integrated'],
        )
        sample_comp_pct = sample_comp.div(sample_comp.sum(axis=1), axis=0) * 100
        
        print(f'Bhaduri — {len(sample_comp)} samples')
        print()
        print('Cell type composition per sample (%):')
        print(sample_comp_pct.round(1).to_string())
        print()
        print('Cell counts per sample:')
        print(sample_comp.sum(axis=1).sort_values(ascending=False))
    else:
        print('No sample values mapped for Bhaduri cells')
else:
    print('sample column not available')

In [ ]:
# Bhaduri sample on UMAP
if 'sample' in adata.obs.columns and adata.obs['sample'].notna().sum() > 0:
    sc.pl.umap(adata, color='sample', title='Integrated UMAP — Bhaduri sample')
else:
    print('sample not available for UMAP plot')

## 16. Save Annotated Object

In [ ]:
# Save the annotated AnnData to Drive
# New columns added: cell_type_integrated, sample, gestational_week
# Also contains: rank_genes_groups results in adata.uns

adata.write_h5ad(PATHS['annotated'])

size_mb = os.path.getsize(PATHS['annotated']) / 1e6
print(f'Saved: {PATHS["annotated"]} ({size_mb:.1f} MB)')
print()
print('Object summary:')
print(f'  Cells:               {adata.shape[0]:,}')
print(f'  Genes (HVG set):     {adata.shape[1]:,}')
print(f'  Genes (raw):         {adata.raw.shape[1]:,}')
print(f'  Leiden clusters:     {adata.obs["leiden"].nunique()}')
print(f'  Cell types:          {adata.obs["cell_type_integrated"].nunique()}')
print(f'  obs keys:            {list(adata.obs.columns)}')
print(f'  obsm keys:           {list(adata.obsm.keys())}')
print(f'  uns keys:            {[k for k in adata.uns.keys()]}')

## Done

The integrated object now has:
- **`cell_type_integrated`** — new cell type labels for all 19 Leiden clusters, annotated in the shared Harmony embedding
- **`gestational_week`** — re-added for Zhong cells (NaN for Bhaduri)
- **`sample`** — re-added for Bhaduri cells (NaN for Zhong)
- **`rank_genes_groups`** — Wilcoxon marker genes per integrated cluster stored in `adata.uns`

Saved to Drive: `data/processed/integrated/integrated_annotated.h5ad`

**Key outputs to inspect:**
- Section 6 dotplot: do clusters express the expected canonical markers?
- Section 10 annotation: are all 19 clusters assigned a biologically plausible label?
- Section 13 composition: which cell types are shared vs dataset-specific?
- Section 14 temporal: does fetal cell type composition shift with gestational age as expected?

**Next steps:**
- `colab_05_trajectory.ipynb` — pseudotime analysis (Monocle3 / diffusion pseudotime) on the annotated integrated object
- `colab_06_rna_velocity.ipynb` — scVelo (requires spliced/unspliced count matrices)